In [ ]:
# Locate the repository when Jupyter starts in a notebook subdirectory.
from pathlib import Path
import sys

_start = Path.cwd().resolve()
_repo = next((p for p in (_start, *_start.parents)
              if (p / "figure" / "paths.py").is_file()
              and (p / "run_cross_validation.py").is_file()), None)
if _repo is None:
    raise RuntimeError("Open this notebook inside the cloned sAge repository.")
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))
from figure.paths import input_path, output_path, font_path


# data- preprocess

Prepare Tabula Sapiens expression data and metadata for the human analyses.

Run Jupyter from the repository root. Required external data and results are listed in `figure/INPUTS.md`. Set `SAGE_FIGURE_INPUT_ROOT` and `SAGE_FIGURE_OUTPUT_ROOT` when using other directories. See figure/VALIDATION.md for the execution checks and their limits.


In [ ]:
import scanpy as sc
import pandas as pd
import re


file_path = input_path("2-8.3-shanda/1-data/GSE201333_RAW/GSM6058681_TabulaSapiens_filtered.h5ad")
adata = sc.read_h5ad(file_path)
print(f"原始基因数量: {adata.n_vars}")




clone_pattern = r'^[A-Z]{2}\d+\.\d+$'



is_clone = adata.var_names.str.contains(clone_pattern, regex=True)


n_clones = is_clone.sum()
print(f"检测到克隆号/未命名基因数量: {n_clones}")
if n_clones > 0:
    print(f"删除示例: {adata.var_names[is_clone][:5].tolist()}")


adata_filtered = adata[:, ~is_clone].copy()

print(f"过滤后剩余基因数量: {adata_filtered.n_vars}")


output_h5ad = input_path("2-8.3-shanda/1-data/GSE201333_RAW/GSM6058681_TabulaSapiens_no_clones.h5ad")
adata_filtered.write_h5ad(output_h5ad)


output_txt = input_path("2-8.3-shanda/1-data/GSE201333_RAW/gene_names_no_clones.txt")
pd.Series(adata_filtered.var_names).to_csv(output_txt, index=False, header=False)

print(f"✅ 任务完成！")
print(f"新的数据集已保存至: {output_h5ad}")
print(f"新的基因列表已保存至: {output_txt}")

In [ ]:
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad(input_path("2-8.3-shanda/1-data/GSE201333_RAW/GSM6058681_TabulaSapiens_no_clones.h5ad"))
adata

In [ ]:

unique_organs = adata.obs['age'].unique()

print(unique_organs)

number_of_organs = unique_organs.size

print(f"Number of unique organs: {number_of_organs}")

In [ ]:

unique_organs = adata.obs['organ_tissue'].unique()

print(unique_organs)

number_of_organs = unique_organs.size

print(f"Number of unique organs: {number_of_organs}")

In [ ]:


unique_organs = adata.obs['method'].unique()

print(unique_organs)

number_of_organs = unique_organs.size

print(f"Number of unique organs: {number_of_organs}")

In [ ]:
import scanpy as sc
import h5py
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy import sparse
import os


INPUT_H5AD = input_path("2-8.3-shanda/1-data/GSE201333_RAW/GSM6058681_TabulaSapiens_no_clones.h5ad")
OUTPUT_DIR = input_path("2-8.3-shanda/1-data/GSE201333_RAW/")
OUTPUT_FILENAME = "2-human-tissue-filter-remove-AL-all-tissue.hdf5"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

CHUNK_SIZE = 50000
COMPRESSION_OPTS = "lzf"

COL_AGE = 'age'
COL_METHOD = 'method'
COL_TISSUE = 'organ_tissue'
# ===========================================

def convert_fast():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Loading metadata from {INPUT_H5AD}...")
    adata = sc.read_h5ad(INPUT_H5AD, backed='r')
    n_obs = adata.n_obs
    n_vars = adata.n_vars


    print("Processing labels...")
    obs_df = adata.obs[[COL_AGE, COL_METHOD, COL_TISSUE]].copy()




    age_cats = pd.Categorical(obs_df[COL_AGE])
    labels_age = age_cats.codes.astype(np.float32)



    age_categories = np.array(age_cats.categories.astype(str), dtype='S')

    print(f" -> Age mapped to range: 0 - {int(labels_age.max())}")
    print(f" -> Total unique age groups: {len(age_categories)}")
    # ======================================================

    # Method
    if COL_METHOD in obs_df:
        method_cats = pd.Categorical(obs_df[COL_METHOD])
        labels_method = method_cats.codes.astype(np.float32)
        method_names = np.array(method_cats.categories.astype(str), dtype='S')
    else:
        labels_method = np.zeros(n_obs, dtype=np.float32)
        method_names = np.array([], dtype='S')

    # Tissue
    if COL_TISSUE in obs_df:
        tissue_cats = pd.Categorical(obs_df[COL_TISSUE])
        labels_tissue = tissue_cats.codes.astype(np.float32)
        tissue_names = np.array(tissue_cats.categories.astype(str), dtype='S')
    else:
        labels_tissue = np.zeros(n_obs, dtype=np.float32)
        tissue_names = np.array([], dtype='S')

    label_matrix = np.column_stack((labels_age, labels_method, labels_tissue))


    print(f"Writing to {OUTPUT_PATH} (Compression: {COMPRESSION_OPTS})...")

    with h5py.File(OUTPUT_PATH, "w") as f:

        dset_data = f.create_dataset("data", shape=(n_obs, n_vars),
                                     dtype='float32',
                                     compression=COMPRESSION_OPTS)

        dset_label = f.create_dataset("label", shape=(n_obs, 3), dtype='float32')
        dset_label[:] = label_matrix



        if len(age_categories) > 0: f.attrs['age_categories'] = age_categories
        if len(method_names) > 0: f.attrs['method_names'] = method_names
        if len(tissue_names) > 0: f.attrs['tissue_names'] = tissue_names


        print(f"Writing data in chunks of {CHUNK_SIZE}...")

        X_source = adata.X

        for i in tqdm(range(0, n_obs, CHUNK_SIZE), desc="Fast Converting"):
            end = min(i + CHUNK_SIZE, n_obs)
            chunk = X_source[i:end]

            if sparse.issparse(chunk):
                chunk = chunk.toarray()

            dset_data[i:end] = chunk.astype(np.float32, copy=False)

    print("\nDone! Fast conversion finished.")
    print("Mapping info saved in HDF5 attributes (age_categories).")

if __name__ == "__main__":
    convert_fast()

In [ ]:
import h5py
import numpy as np

file_path = input_path("2-8.3-shanda/1-data/GSE201333_RAW/2-human-tissue-filter-remove-AL-all-tissue.hdf5")

with h5py.File(file_path, "r") as f:

    print("数据集列表:", list(f.keys()))


    if 'label' in f:
        labels = f['label'][:]
        print("\nLabel数据信息:")
        print("形状:", labels.shape)
        print("数据类型:", labels.dtype)
        print("前5行数据:\n", labels[:5])


        for col in range(labels.shape[1]):
            unique_values = np.unique(labels[:, col])
            print(f"列{col}的唯一值({len(unique_values)}个): {unique_values}")


    if 'data' in f:
        data = f['data']
        print("\nData数据信息:")
        print("形状:", data.shape)
        print("数据类型:", data.dtype)


        print("数据示例(前3行前10列):\n", data[:3, :10] if len(data.shape) == 2 else "非2维数据")


In [ ]:
import h5py
import numpy as np
import os
import sys
from tqdm import tqdm



SOURCE_H5_PATH = input_path(r"2-8.3-shanda/1-data/GSE201333_RAW/2-human-tissue-filter-remove-AL-all-tissue.hdf5")

OUTPUT_DIR = input_path(r"2-8.3-shanda/1-data/GSE201333_RAW/2-human-tissue-filter-remove-AL-all-tissue")


BATCH_SIZE = 50000
# ===========================================

def split_hdf5_dynamic():

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"--- 开始按细胞类型拆分数据 (动态映射模式) ---")
    print(f"源文件: {SOURCE_H5_PATH}")
    print(f"输出目录: {OUTPUT_DIR}")

    if not os.path.exists(SOURCE_H5_PATH):
        print(f"错误: 找不到源文件 {SOURCE_H5_PATH}")
        return

    try:

        with h5py.File(SOURCE_H5_PATH, "r") as f_src:

            data_src = f_src['data']
            label_src = f_src['label']

            total_samples = data_src.shape[0]
            num_features = data_src.shape[1]
            label_dim = label_src.shape[1]

            print(f"总样本数: {total_samples:,}, 特征数: {num_features:,}")


            print("正在读取 'tissue_names' 属性以构建映射表...")
            if 'tissue_names' not in f_src.attrs:
                print("错误: 源文件中缺少 'tissue_names' 属性，无法获取细胞类型名称。")
                return


            raw_names = f_src.attrs['tissue_names']
            cell_type_map = {}
            for idx, name in enumerate(raw_names):

                if isinstance(name, bytes):
                    name_str = name.decode('utf-8')
                else:
                    name_str = str(name)
                cell_type_map[idx] = name_str

            print(f" -> 成功加载 {len(cell_type_map)} 个细胞类型定义。")


            print("正在读取 Label 矩阵以统计细胞分布...")

            cell_type_column = label_src[:, 2].astype(int)


            present_ids = np.unique(cell_type_column)
            print(f" -> 数据中实际包含 {len(present_ids)} 种细胞类型。")


            global_pbar = tqdm(present_ids, desc="总体进度", unit="type")

            for type_id in global_pbar:

                type_name = cell_type_map.get(type_id, f"type_{type_id}")


                safe_name = type_name.replace(" ", "_").replace("/", "_").replace(",", "").lower()


                global_pbar.set_description(f"处理: ID {type_id} ({safe_name})")


                type_indices = np.where(cell_type_column == type_id)[0]
                num_type_samples = len(type_indices)

                if num_type_samples == 0:
                    continue



                output_filename = f"type_{type_id}_{safe_name}.hdf5"
                output_path = os.path.join(OUTPUT_DIR, output_filename)


                with h5py.File(output_path, "w") as f_dst:

                    dset_data = f_dst.create_dataset('data',
                                                     shape=(num_type_samples, num_features),
                                                     dtype=data_src.dtype,
                                                     chunks=True,
                                                     compression="lzf")


                    dset_label = f_dst.create_dataset('label',
                                                      shape=(num_type_samples, label_dim),
                                                      dtype=label_src.dtype,
                                                      chunks=True,
                                                      compression="lzf")


                    f_dst.attrs['original_type_id'] = type_id
                    f_dst.attrs['original_type_name'] = type_name

                    if 'age_categories' in f_src.attrs:
                        f_dst.attrs['age_categories'] = f_src.attrs['age_categories']



                    sorted_indices = np.sort(type_indices)

                    for start in tqdm(range(0, num_type_samples, BATCH_SIZE),
                                      desc=f" -> 写入 {safe_name}",
                                      leave=False):
                        end = min(start + BATCH_SIZE, num_type_samples)


                        batch_global_indices = sorted_indices[start:end]


                        batch_data = data_src[batch_global_indices]
                        batch_label = label_src[batch_global_indices]


                        dset_data[start:end] = batch_data
                        dset_label[start:end] = batch_label

    except Exception as e:
        print(f"\n发生意外错误: {e}")
        import traceback
        traceback.print_exc()

    print(f"\n--- 拆分完成 ---")
    print(f"文件已保存在: {OUTPUT_DIR}")

if __name__ == "__main__":
    split_hdf5_dynamic()

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os


INPUT_H5AD = input_path("2-8.3-shanda/1-data/GSE201333_RAW/GSM6058681_TabulaSapiens_no_clones.h5ad")
OUTPUT_TXT = input_path("2-8.3-shanda/1-data/GSE201333_RAW/4-human-tissue-filter-remove-AC-all-tissue.txt")


COL_AGE = 'age'
COL_METHOD = 'method'
COL_TISSUE = 'organ_tissue'
# ===========================================

def generate_summary():
    print(f"Reading metadata from {INPUT_H5AD}...")

    adata = sc.read_h5ad(INPUT_H5AD, backed='r')

    n_obs = adata.n_obs
    n_vars = adata.n_vars
    filename = os.path.basename(INPUT_H5AD)
    dataset_name = filename.replace('.h5ad', '')


    obs_df = adata.obs[[COL_AGE, COL_METHOD, COL_TISSUE]].copy()

    print(f"Generating summary to {OUTPUT_TXT}...")

    with open(OUTPUT_TXT, 'w', encoding='utf-8') as f:

        f.write("data:\n")
        f.write(f"Data shape for {filename}: ({n_obs}, {n_vars})\n")
        f.write(f"Total: ({n_obs}, {n_vars})\n\n")


        f.write("label:\n")
        f.write(f"Data shape for {dataset_name}: ({n_obs}, 3)\n")
        f.write(f"Total: ({n_obs}, 3)\n\n")


        f.write("age:\n")
        age_counts = obs_df[COL_AGE].value_counts().sort_index()
        for i, (age_val, count) in enumerate(age_counts.items()):
            try:
                age_display = int(age_val)
            except:
                age_display = age_val

            f.write(f"{age_display}: {i} (n={count})\n")
        f.write("\n")


        f.write("method:\n")
        if COL_METHOD in obs_df:
            method_counts = obs_df[COL_METHOD].value_counts().sort_index()
            for i, (method, count) in enumerate(method_counts.items()):
                f.write(f"{method}: {i} (n={count})\n")
        f.write("\n")



        f.write("tissue:\n")
        if COL_TISSUE in obs_df:


            tissue_counts = obs_df[COL_TISSUE].value_counts().sort_index()

            for i, (tissue_name, count) in enumerate(tissue_counts.items()):

                f.write(f"'{tissue_name}': {i} (sample_count: {count})\n")
        else:
            f.write("No tissue/cell_type column found.\n")
        f.write("\n")
        # ===============================================================

    print("Done! Summary file generated with cell type counts.")
    print(f"Path: {OUTPUT_TXT}")

if __name__ == "__main__":
    generate_summary()